# VLM-DENTAL — Stage 2: Group Relative Policy Optimization (GRPO)

This notebook optimizes **Qwen3.5-9B** using GRPO reinforcement learning against clinical ground truth.

### Core Architectural Invariants:
- **Hardware Optimization**: Native support for **Google Cloud TPU v5e-8** (8-way FSDPv2) and multi-GPU clusters.
- **[G2] Batched Rollout Generation**: Batches all $K$ candidate trajectories into a single forward pass, eliminating the sequential decode latency gap.
- **[G3] Dual-LoRA Adapter Toggle**: Attaches frozen Stage 1 SFT reference (`"reference"`) and trainable RL policy (`"grpo_policy"`) to a single base model in memory.
- **Flexible $K \in \{1, 2, 4, 8, 16\}$ Sweep**: Evaluates group-relative advantage normalization and EMA fallback for $K=1$.
- **Cross-Turn KV-Cache Reuse**: Preserves `DynamicCache` across agent turns with 3D MRoPE coordinate slicing, bypassing the vision encoder on metadata returns.
- **Multi-Finding Reward Matching (Rule 13)**: Full set-level bipartite matching across all findings per image.
- **Hugging Face Hub Checkpoint Sync**: Uploads lightweight checkpoints (~760 MB) every 25 steps to survive Kaggle 9h session timeouts and enable cross-account resume.

## 1. Hardware Auto-Detection (TPU v5e-8 vs GPU)

In [ ]:
import os
import sys
import torch

IS_TPU = False
DEVICE_STR = "cpu"

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    IS_TPU = True
    DEVICE_STR = f"TPU ({xm.xla_device_hw(device)}) - {device}"
    print(f"[HARDWARE] Detected Cloud TPU: {DEVICE_STR}")
except Exception:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()
        DEVICE_STR = f"GPU ({gpu_count}x {gpu_name})"
        print(f"[HARDWARE] Detected CUDA: {DEVICE_STR}")
    else:
        print("[HARDWARE] Running on CPU (Testing only).")

print(f"PyTorch Version: {torch.__version__}")

## 2. Environment Setup & Dependency Installation

In [ ]:
# Clone or pull latest VLM-DENTAL codebase
if not os.path.exists("VLM-DENTAL"):
    !git clone https://github.com/rezaxr14/VLM-DENTAL.git
    %cd VLM-DENTAL
else:
    %cd VLM-DENTAL
    !git pull

# Install project and RL dependencies
!pip install -q -e .
!pip install -q peft trl datasets accelerate huggingface_hub ultralytics python-dotenv pytest
!pip install -q qwen-vl-utils

## 3. Hugging Face Authentication & Reference SFT Checkpoint

In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[AUTH] Logged into Hugging Face via HF_TOKEN.")
else:
    print("[AUTH] Please log into Hugging Face:")
    login()

# Track selection determines the required Stage 1 SFT reference model
TRACK = "with_tools"  # "with_tools" or "no_tools"
sft_tag = "qwen3_5_9b_sft_tools" if TRACK == "with_tools" else "qwen3_5_9b_sft_no_tools"
sft_dir = f"data/models/{sft_tag}"

if not os.path.exists(os.path.join(sft_dir, "adapter_model.safetensors")):
    print(f"[SYNC] Local SFT adapter not found at {sft_dir}. Pulling from Hugging Face Hub...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Reza-Nadimi/vlm-dental-checkpoints",
        allow_patterns=[f"{sft_tag}/*"],
        local_dir="data/models"
    )
    print("[SYNC] SFT reference checkpoint restored successfully.")
else:
    print(f"[SYNC] SFT reference checkpoint verified locally at {sft_dir}.")

## 4. [G3] Dual-LoRA Reference/Policy Verification Test

In [ ]:
# Run explicit unit verification proving that adapter toggling works and KL divergence is non-negative
print("[VERIFY] Executing G3 Dual-LoRA Adapter Toggle Test...")
!python -m pytest tests/test_dual_adapter_grpo.py -v

## 5. GRPO Training Configuration & Sweep Options

In [ ]:
# =========================================================================
# STAGE 2 GRPO EXECUTION PARAMETERS
# =========================================================================
# Execution Mode: "single" (single K run) or "sweep" (sweeps K in [1, 2, 4, 8, 16])
MODE = "single"

# Group Size (for "single" mode): 1, 2, 4, 8, or 16
GROUP_SIZE = 4

# Hugging Face Checkpoint Repository
HF_REPO = "Reza-Nadimi/vlm-dental-checkpoints"

# Auto-upload checkpoint every N steps
PUSH_EVERY_STEPS = 25

# Set RESUME = True if continuing from a previously uploaded HF checkpoint across Kaggle accounts
RESUME = False

print(f"[CONFIG] Track: {TRACK}")
print(f"[CONFIG] Mode: {MODE} (Group Size K={GROUP_SIZE} if single)")
print(f"[CONFIG] HF Checkpoint Hub: {HF_REPO}")
print(f"[CONFIG] Auto-upload every: {PUSH_EVERY_STEPS} steps")

## 6. Launch GRPO Training / Sweep Pipeline

In [ ]:
if MODE == "sweep":
    print("[EXECUTE] Launching automated K in [1, 2, 4, 8, 16] sweep orchestrator...")
    cmd = [
        "python", "scripts/run_grpo_sweep.py",
        "--track", TRACK,
        "--hf-repo", HF_REPO,
        "--push-every-steps", str(PUSH_EVERY_STEPS)
    ]
else:
    print(f"[EXECUTE] Launching GRPO training with group size K={GROUP_SIZE}...")
    cmd = [
        "python", "scripts/run_grpo.py",
        "--track", TRACK,
        "--group-size", str(GROUP_SIZE),
        "--epochs", "2",
        "--lr", "5e-6",
        "--kl-beta", "0.04",
        "--hf-repo", HF_REPO,
        "--push-every-steps", str(PUSH_EVERY_STEPS)
    ]
    if RESUME:
        cmd.extend(["--resume-hf", HF_REPO])

cmd_str = " ".join(cmd)
print(f"[EXECUTE] Running: {cmd_str}")
!{cmd_str}

## 7. Real-Time RL Reward & KL Convergence Dashboard

In [ ]:
import json
import matplotlib.pyplot as plt

log_file = "data/eval_results/grpo_training_log.jsonl"
if os.path.exists(log_file):
    steps, rewards, kls = [], [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                steps.append(r.get("step", len(steps)))
                rewards.append(r.get("mean_reward", 0.0))
                kls.append(r.get("kl_divergence", 0.0))
                
    fig, ax1 = plt.subplots(figsize=(11, 5))
    
    color = "#2ca02c"
    ax1.set_xlabel("RL Optimization Steps")
    ax1.set_ylabel("Mean Trajectory Reward", color=color)
    ax1.plot(steps, rewards, color=color, lw=2, label="Mean Reward")
    ax1.tick_params(axis="y", labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    ax2 = ax1.twinx()
    color = "#d62728"
    ax2.set_ylabel("KL Divergence (Schulman k3)", color=color)
    ax2.plot(steps, kls, color=color, lw=1.5, ls="--", label="KL Divergence")
    ax2.tick_params(axis="y", labelcolor=color)
    
    plt.title(f"VLM-DENTAL Stage 2 GRPO RL Training Progress (K={GROUP_SIZE})")
    fig.tight_layout()
    plt.show()
    print(f"[RESULTS] Current Reward: {rewards[-1]:.4f} | Current KL: {kls[-1]:.4f}")
else:
    print(f"[INFO] Log file {log_file} not yet generated. Run training above to view RL curves.")